
# Phase 5-4: Zero-Shot 추론 및 시각화 (XGBoost 버전)
이 노트북은 앞서 훈련된 `universal_xgb_model.pkl` (XGBoost)을 불러와서 
동구(Dong-gu) 지역 격자망에 추론을 수행하고 핫스팟 지도를 시각화합니다.
또한 `rules.md`의 [Rule 1], [Rule 5]를 준수하여 후처리(Post-processing)를 수행합니다.


In [ ]:

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import warnings
warnings.filterwarnings('ignore')

base_dir = Path('../../')
model_path = base_dir / 'data/models/universal_xgb_model.pkl'

# 동구 도화지 사용 (비교 검증용)
data_path = base_dir / 'data/processed/dataset_dong-gu_gwangju_south_korea_10m_buf10m_baseline.gpkg'

import xgboost as xgb
class PUBaggingXGBoost:
    def __init__(self, n_estimators=10, random_state=42):
        self.n_estimators = n_estimators
        self.models = []
        self.random_state = random_state
    def fit(self, X, y): pass
    def predict_proba(self, X):
        probas = np.array([model.predict_proba(X)[:, 1] for model in self.models])
        return probas.mean(axis=0)

print("✅ 라이브러리 및 모델 래퍼 로드 완료")


In [ ]:

print(f"📦 XGBoost 모델 로드 중... ({model_path.name})")
model = joblib.load(model_path)

print(f"🗺️ 타겟 데이터 로드 중... ({data_path.name})")
gdf = gpd.read_file(data_path)

feature_cols = [col for col in gdf.columns if '_count_' in col]
X = gdf[feature_cols].values

print(f"추출된 피처 ({len(feature_cols)}개): {feature_cols}")



## [Rule 5 준수] 추론 및 점수 정규화 (NaN 방어)
GraphHopper 백엔드 엔진이 `IllegalArgumentException`을 던지지 않도록 
점수 클리핑과 결측치 처리를 강제합니다.


In [ ]:

print("🧠 추론(Inference) 진행 중...")
trash_score = model.predict_proba(X)

# 🛑 [Rule 5: Backend Integration Compliance]
# 1. NaN 값을 0.0으로 치환
trash_score = np.nan_to_num(trash_score, nan=0.0)
# 2. 모든 점수를 0.0과 1.0 사이로 완벽히 클리핑
trash_score = np.clip(trash_score, 0.0, 1.0)

gdf['trash_score'] = trash_score
print(f"✅ 정규화된 점수 분포: 최소 {trash_score.min():.4f} ~ 최대 {trash_score.max():.4f}")



## [Rule 1 & 5 준수] Geographic CRS 투영 (PostGIS/GraphHopper 대비)
동구 데이터는 현재 미터법 단위의 투영좌표계(EPSG:5179)이므로, 
라우팅 엔진에서 위경도로 파악할 수 있도록 지리좌표계(EPSG:4326)로 변환합니다.


In [ ]:

print(f"현재 CRS: {gdf.crs}")
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(epsg=4326)
    print("✅ [Rule 1 & 5] Geographic CRS (EPSG:4326) 변환 완료!")



## 시각화
XGBoost의 강력한 의사결정 경계를 시각적으로 확인합니다.


In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(15, 15))

gdf.plot(
    column='trash_score',
    ax=ax,
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': "Trash Spot Probability (XGBoost)", 'orientation': "horizontal", 'shrink': 0.6},
    alpha=0.9,
    edgecolor='none'
)

ax.set_facecolor('#1e1e1e')
fig.patch.set_facecolor('#1e1e1e')
ax.set_title("Hotspot Prediction Map (XGBoost)", color='white', fontsize=18, pad=20)
ax.axis('off')
if ax.get_legend(): plt.setp(ax.get_legend().get_texts(), color='white')

plt.show()
